In [3]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import optuna
import math
import time
import random
from collections import defaultdict
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.model_selection import cross_val_score, KFold, train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# Tắt các cảnh báo không cần thiết
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings("ignore")

In [4]:
# ============================================================================
# BƯỚC 1: ĐỊNH NGHĨA KHÔNG GIAN TÌM KIẾM (SEARCH SPACES)
# Định nghĩa các không gian tìm kiếm cho cả 4 mô hình.
# ============================================================================

MASTER_SEARCH_SPACES = {
    'logistic_regression': {
        'classifier__C': ('float', 0.01, 100.0, 'log'),
        'classifier__solver': ('categorical', ['liblinear', 'saga'])
    },
    'random_forest': {
        'classifier__n_estimators': ('int', 100, 1000),
        'classifier__max_depth': ('int', 5, 50),
        'classifier__min_samples_split': ('int', 2, 10),
        'classifier__min_samples_leaf': ('int', 1, 5)
    },
    'xgboost': {
        'classifier__learning_rate': ('float', 0.01, 0.3, 'log'),
        'classifier__n_estimators': ('int', 100, 1000),
        'classifier__max_depth': ('int', 3, 10),
        'classifier__subsample': ('float', 0.6, 1.0),
        'classifier__colsample_bytree': ('float', 0.6, 1.0),
        'classifier__gamma': ('float', 0, 5)
    },
    'lightgbm': {
        'classifier__learning_rate': ('float', 0.01, 0.3, 'log'),
        'classifier__n_estimators': ('int', 100, 1000),
        'classifier__num_leaves': ('int', 20, 150),
        'classifier__max_depth': ('int', 3, 10),
        'classifier__subsample': ('float', 0.6, 1.0),
        'classifier__colsample_bytree': ('float', 0.6, 1.0)
    }
}

In [5]:
# ============================================================================
# BƯỚC 2: TẢI VÀ TIỀN XỬ LÝ DỮ LIỆU
# Một hàm chung để tải và chuẩn bị 4 bộ dữ liệu.
# ============================================================================

def get_data(dataset_name):
    """
    Tải và tiền xử lý một trong 4 bộ dữ liệu.
    Trả về (X, y, preprocessor, metric).
    """
    if dataset_name == 'breast_cancer':
        print("... Đang tải Breast Cancer")
        X, y = load_breast_cancer(return_X_y=True)
        # Dữ liệu này đã sạch và chỉ có số
        preprocessor = StandardScaler()
        metric = 'accuracy'
        return X, y, preprocessor, metric

    elif dataset_name == 'adult':
        print("... Đang tải Adult Income")
        # Dữ liệu này cần được tải từ file và có các cột categorical
        # (Giả sử bạn có 'adult.csv' trong cùng thư mục)
        try:
            url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
            columns = [
                'age', 'workclass', 'fnlwgt', 'education', 'education-num',
                'marital-status', 'occupation', 'relationship', 'race', 'sex',
                'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
            ]
            data = pd.read_csv(url, header=None, names=columns, na_values='?', skipinitialspace=True)
        except Exception as e:
            print(f"Lỗi khi tải Adult dataset: {e}")
            print("Vui lòng tải 'adult.data' từ UCI và đổi tên thành 'adult.csv'")
            return None, None, None, None

        data = data.dropna()
        X = data.drop('income', axis=1)
        y = data['income'].map({'<=50K': 0, '>50K': 1}).values

        # Định nghĩa các cột số và categorical
        numeric_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
        categorical_features = ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

        # Tạo preprocessor pipeline
        numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
        categorical_transformer = Pipeline(steps=[('encoder', OneHotEncoder(handle_unknown='ignore'))])
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features),
                ('cat', categorical_transformer, categorical_features)
            ])
        metric = 'accuracy'
        return X, y, preprocessor, metric

    elif dataset_name == 'telco_churn':
        print("... Đang tải Telco Customer Churn")
        # (Giả sử bạn có 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
        try:
            data = pd.read_csv(r'C:\Users\nguye\Downloads\AMSCO\datasets\telco.csv')
        except FileNotFoundError:
            print("Vui lòng tải 'WA_Fn-UseC_-Telco-Customer-Churn.csv' từ Kaggle.")
            return None, None, None, None
        
        data = pd.to_numeric(data, errors='coerce')
        data = data.dropna()
        data = data.drop('customerID', axis=1)

        X = data.drop('Churn', axis=1)
        y = data['Churn'].map({'No': 0, 'Yes': 1}).values

        numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
        categorical_features = [col for col in X.columns if col not in numeric_features]

        numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
        categorical_transformer = Pipeline(steps=[('encoder', OneHotEncoder(handle_unknown='ignore'))])


        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features),
                ('cat', categorical_transformer, categorical_features)
            ])
        metric = 'accuracy'
        return X, y, preprocessor, metric

    elif dataset_name == 'credit_card_fraud':
        print("... Đang tải Credit Card Fraud")
        # (Giả sử bạn có 'creditcard.csv')
        try:
            data = pd.read_csv(r'C:\Users\nguye\Downloads\AMSCO\datasets\creditcard.csv')
        except FileNotFoundError:
            print("Vui lòng tải 'creditcard.csv' từ Kaggle.")
            return None, None, None, None

        # Dữ liệu này bị mất cân bằng nghiêm trọng
        # và đã được PCA, chỉ cần scale 'Time' và 'Amount'
        X = data.drop('Class', axis=1)
        y = data['Class'].values
        
        # Chỉ scale 2 cột không phải PCA
        numeric_features = ['Time', 'Amount']
        passthrough_features = [col for col in X.columns if col not in numeric_features]

        numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
        
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features),
                ('pass', 'passthrough', passthrough_features)
            ])
        
        # Do mất cân bằng, 'roc_auc' là metric tốt hơn 'accuracy' [9, 10]
        metric = 'roc_auc' 
        return X, y, preprocessor, metric

    else:
        raise ValueError(f"Bộ dữ liệu '{dataset_name}' không được hỗ trợ.")

In [6]:
# ============================================================================
# BƯỚC 3: HÀM MỤC TIÊU (OBJECTIVE FUNCTION) TỔNG QUÁT
# ============================================================================

def create_objective(X, y, model_name, preprocessor, metric='accuracy'):
    """
    Tạo ra một hàm mục tiêu (objective) để truyền vào các HPO optimizer.
    """
    
    def objective(params):
        # 1. Định nghĩa mô hình dựa trên model_name
        if model_name == 'logistic_regression':
            model = LogisticRegression(random_state=42, max_iter=1000)
        elif model_name == 'random_forest':
            model = RandomForestClassifier(random_state=42)
        elif model_name == 'xgboost':
            model = xgb.XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False)
        elif model_name == 'lightgbm':
            model = lgb.LGBMClassifier(random_state=42, verbosity=-1)
        else:
            raise ValueError(f"Mô hình '{model_name}' không được hỗ trợ.")
            
        # 2. Tạo một pipeline hoàn chỉnh
        # Lưu ý: tên 'classifier' phải khớp với tên trong MASTER_SEARCH_SPACES
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
        
        # 3. Gán các siêu tham số cho pipeline
        # Cú pháp `classifier__C` sẽ tự động gán 'C' cho bước 'classifier'
        pipeline.set_params(**params)
        
        # 4. Đánh giá mô hình
        # Đây là 'Inner CV' của chúng ta [11, 12, 13, 14, 15, 16]
        cv = KFold(n_splits=3, shuffle=True, random_state=42)
        
        try:
            score = cross_val_score(pipeline, X, y, cv=cv, scoring=metric).mean()
        except Exception as e:
            print(f"Lỗi khi đánh giá {params}: {e}")
            return 0.0 # Trả về điểm số thấp nếu có lỗi

        return score
    
    return objective

In [7]:
# ============================================================================
# BƯỚC 4: FRAMEWORK AMSCO (PHIÊN BẢN TỔNG QUÁT VÀ ĐÃ SỬA LỖI)
# ============================================================================

class KnowledgeHub:
    def __init__(self):
        self.trials = []
        self.best_score = -float('inf')
        self.best_params = None

    def store(self, agent_id, params, score):
        self.trials.append({'agent_id': agent_id, 'params': params, 'score': score})
        if score > self.best_score:
            self.best_score = score
            self.best_params = params
            # print(f"  [KnowledgeHub] New best score: {self.best_score:.4f} from {agent_id}")

    def get_all_trials(self):
        return self.trials

    def get_best_trial(self):
        return {'params': self.best_params, 'score': self.best_score}

class StrategyAgent:
    """Lớp cơ sở, giờ nhận objective và search_space"""
    def __init__(self, agent_id, objective_func, search_space, knowledge_hub):
        self.agent_id = agent_id
        self.objective = objective_func
        self.search_space = search_space
        self.knowledge_hub = knowledge_hub

    def run(self, budget):
        raise NotImplementedError

class RandomAgent(StrategyAgent):
    """RandomAgent: Giờ đã hoàn toàn linh hoạt"""
    def run(self, budget):
        # print(f"    -> Running RandomAgent with budget: {budget}")
        for _ in range(budget):
            params = {}
            
            for name, details in self.search_space.items():
                type = details[0]  # Type là phần tử đầu tiên của tuple
                if type == 'float':
                    low, high = details[1], details[2]
                    dist_type = details[3] if len(details) > 3 else None
                    if dist_type == 'log':
                        params[name] = np.exp(random.uniform(np.log(low), np.log(high)))
                    else:
                        params[name] = random.uniform(low, high)
                elif type == 'int':
                    low, high = details[1], details[2]
                    params[name] = random.randint(low, high)
                elif type == 'categorical':
                    choices = details[1] # Lấy danh sách lựa chọn
                    params[name] = random.choice(choices)
            
            score = self.objective(params)
            self.knowledge_hub.store(self.agent_id, params, score)

class BayesianAgent(StrategyAgent):
    """BayesianAgent: Giờ đã hoàn toàn linh hoạt"""
    def run(self, budget):
        # print(f"    -> Running BayesianAgent with budget: {budget}")
        
        # (A) Tự động xây dựng hàm objective cho Optuna
        def optuna_objective(trial):
            params = {}
            for name, details in self.search_space.items():
                type = details[0]  # Type là phần tử đầu tiên của tuple
                if type == 'float':
                    low, high = details[1], details[2]
                    dist_type = details[3] if len(details) > 3 else None
                    params[name] = trial.suggest_float(name, low, high, log=(dist_type == 'log'))
                elif type == 'int':
                    low, high = details[1], details[2]
                    params[name] = trial.suggest_int(name, low, high)
                elif type == 'categorical':
                    choices = details[1]
                    params[name] = trial.suggest_categorical(name, choices)

            score = self.objective(params)
            self.knowledge_hub.store(self.agent_id, params, score) # Vẫn báo cáo về Hub chung
            return score

        # (B) Khởi tạo và Warm-start
        study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
        
        existing_trials = self.knowledge_hub.get_all_trials()
        if existing_trials:
            # print(f"  ... BayesianAgent warm-starting with {len(existing_trials)} previous trials.")
            
            # Tự động tạo 'distributions' cho Optuna
            distributions = {}
            for name, details in self.search_space.items():
                type = details[0]  # Type là phần tử đầu tiên của tuple
                if type == 'float':
                    low, high = details[1], details[2]
                    dist_type = details[3] if len(details) > 3 else None
                    distributions[name] = optuna.distributions.FloatDistribution(low, high, log=(dist_type == 'log'))
                elif type == 'int':
                    low, high = details[1], details[2]
                    distributions[name] = optuna.distributions.IntDistribution(low, high)
                elif type == 'categorical':
                    choices = details[1]
                    distributions[name] = optuna.distributions.CategoricalDistribution(choices)

            for t in existing_trials:
                if not t or 'params' not in t or 'score' not in t:
                    continue  # Bỏ qua trial không hợp lệ
                    
                try:
                    # Đảm bảo các tham số từ agent khác nằm trong không gian
                    valid_params = {}
                    for k, v in t['params'].items():
                        if k in distributions:
                            # Đặc biệt xử lý cho categorical parameters
                            if isinstance(distributions[k], optuna.distributions.CategoricalDistribution):
                                if v in distributions[k].choices:
                                    valid_params[k] = v
                            else:
                                valid_params[k] = v
                    
                    if not valid_params or len(valid_params) != len(t['params']):
                        continue  # Bỏ qua trial có tham số lạ hoặc giá trị không hợp lệ
                    
                    # Đảm bảo score là một số hợp lệ
                    if not isinstance(t['score'], (int, float)) or math.isnan(t['score']):
                        continue
                        
                    frozen_trial = optuna.trial.create_trial(
                        params=valid_params,
                        distributions=distributions,
                        value=t['score']
                    )
                    study.add_trial(frozen_trial)
                except Exception as e:
                    print(f"Lỗi khi thêm trial: {e}")  # In ra lỗi để debug
                    continue  # Bỏ qua các trial không tương thích

        study.optimize(optuna_objective, n_trials=budget, show_progress_bar=False)

class GridAgent(StrategyAgent):
    """GridAgent: Linh hoạt (tinh chỉnh 2 tham số quan trọng nhất)"""
    def run(self, budget):
        # print(f"    -> Running GridAgent with budget: {budget}")
        best_params = self.knowledge_hub.get_best_trial()['params']
        if not best_params:
            # print("  ... GridAgent skipped (no best params yet).")
            return
        
        # Chọn 2 tham số đầu tiên (hoặc float/int) để tinh chỉnh
        params_to_tune = []
        for name, details in self.search_space.items():
            if details[0] in ['float', 'int']:  # Kiểm tra type ở vị trí đầu tiên của tuple
                params_to_tune.append(name)
            if len(params_to_tune) >= 2:
                break
        
        if len(params_to_tune) == 0:
            return 
        
        local_grid = [best_params]
        
        # Lấy tham số đầu tiên để tinh chỉnh
        p1_name = params_to_tune[0]
        p1_val = best_params.get(p1_name)
        
        p1_details = self.search_space[p1_name]
        p1_type = p1_details[0]  # Type là phần tử đầu tiên của tuple
        p1_low = p1_details[1]   # Low là phần tử thứ hai
        p1_high = p1_details[2]  # High là phần tử thứ ba
        # --- KẾT THÚC SỬA LỖI ---
        
        # Tạo 3-5 điểm cho p1
        if p1_type == 'float':
            p1_steps = np.linspace(p1_val * 0.9, p1_val * 1.1, 3)
        else: # int
            p1_steps = {p1_val - 1, p1_val, p1_val + 1}
        
        for p1 in p1_steps:
            # "Kẹp" giá trị trong phạm vi
            p1_clamped = max(p1_low, min(p1_high, p1))
            if p1_type == 'int': p1_clamped = int(p1_clamped)
            
            new_params = {**best_params, p1_name: p1_clamped}
            if new_params not in local_grid:
                local_grid.append(new_params)

        # Giới hạn số lần chạy theo budget
        for i in range(min(budget, len(local_grid))):
            params = local_grid[i]
            score = self.objective(params)
            self.knowledge_hub.store(self.agent_id, params, score)


class PerformanceMonitor:
    def __init__(self, agent_ids):
        self.agent_ids = agent_ids
        self.history = defaultdict(list)

    def update(self, all_trials):
        self.history = defaultdict(list)
        for trial in all_trials:
            self.history[trial['agent_id']].append(trial['score'])

    def get_agent_rewards(self):
        rewards = {}
        for agent_id in self.agent_ids:
            scores = self.history.get(agent_id, [])  # Đặt giá trị mặc định là list rỗng
            if not scores or len(scores) < 2:  # Kiểm tra scores có tồn tại và đủ dài
                rewards[agent_id] = 0.5  # Giá trị mặc định cho agent mới
            else:
                recent_scores = scores[-5:]  # Lấy tối đa 5 điểm gần nhất
                reward = np.mean(np.diff(recent_scores)) if len(recent_scores) > 1 else 0
                normalized_reward = (math.tanh(reward * 100) + 1) / 2
                rewards[agent_id] = normalized_reward
        return rewards

class MetaController_UCB1:
    def __init__(self, agent_ids):
        self.agent_ids = agent_ids
        self.agent_pulls = {agent_id: 0 for agent_id in agent_ids}
        self.agent_rewards = {agent_id: 0.0 for agent_id in agent_ids}
        self.total_pulls = 0

    def allocate(self, slice_budget):
        # Kiểm tra các agent chưa được khởi tạo
        uninitialized_agents = [aid for aid, pulls in self.agent_pulls.items() if pulls == 0]
        if uninitialized_agents:
            agent_to_run = uninitialized_agents[0]  # Lấy agent đầu tiên chưa được khởi tạo
            allocations = {agent_id: 0 for agent_id in self.agent_ids}
            allocations[agent_to_run] = slice_budget
            print(f"  [MetaController] Initializing {agent_to_run}")
            return allocations

        # Tính toán UCB scores cho các agent đã được khởi tạo
        ucb_scores = {}

        ucb_scores = {}
        for agent_id in self.agent_ids:
            if self.agent_pulls[agent_id] == 0:
                ucb_scores[agent_id] = float('inf')
            else:
                avg_reward = self.agent_rewards[agent_id] / self.agent_pulls[agent_id]
                exploration_bonus = math.sqrt(2 * math.log(self.total_pulls) / self.agent_pulls[agent_id])
                ucb_scores[agent_id] = avg_reward + exploration_bonus
        
        best_agent = max(ucb_scores, key=ucb_scores.get)
        print(f"  [MetaController] UCB scores: { {k: f'{v:.2f}' for k, v in ucb_scores.items()} } -> Chose {best_agent}")
        
        allocations = {agent_id: 0 for agent_id in self.agent_ids}
        allocations[best_agent] = slice_budget
        return allocations

    def update(self, agent_id_to_update, reward):
        if reward >= 0:
            self.agent_rewards[agent_id_to_update] += reward
            self.agent_pulls[agent_id_to_update] += 1
            self.total_pulls += 1
            # print(f"  [MetaController] Updated {agent_id_to_update}: pulls={self.agent_pulls[agent_id_to_update]}, total_reward={self.agent_rewards[agent_id_to_update]:.2f}")


class AMSCO_Orchestrator:
    """Orchestrator: Giờ nhận objective và search_space"""
    def __init__(self, objective_func, search_space, total_budget, slice_budget):
        self.total_budget = total_budget
        self.slice_budget = slice_budget
        
        self.knowledge_hub = KnowledgeHub()
        
        self.agents = {
            "Random": RandomAgent("Random", objective_func, search_space, self.knowledge_hub),
            "Bayesian": BayesianAgent("Bayesian", objective_func, search_space, self.knowledge_hub),
            "Grid": GridAgent("Grid", objective_func, search_space, self.knowledge_hub)
        }
        agent_ids = list(self.agents.keys())
        
        self.performance_monitor = PerformanceMonitor(agent_ids)
        self.meta_controller = MetaController_UCB1(agent_ids)

    def run(self):
        current_budget = self.total_budget
        slice_num = 1
        
        while current_budget > 0:
            # print(f"\n--- Slice {slice_num} | Budget remaining: {current_budget} ---")
            
            budget_for_slice = min(self.slice_budget, current_budget)
            
            allocations = self.meta_controller.allocate(budget_for_slice)
            
            for agent_id, budget in allocations.items():
                if budget > 0:
                    # print(f"... Giving budget to {agent_id}")
                    self.agents[agent_id].run(budget)
                    
                    all_trials = self.knowledge_hub.get_all_trials()
                    self.performance_monitor.update(all_trials)
                    
                    reward = self.performance_monitor.get_agent_rewards()[agent_id]
                    self.meta_controller.update(agent_id, reward)
            
            current_budget -= budget_for_slice
            slice_num += 1
            
        final_result = self.knowledge_hub.get_best_trial()
        return final_result

In [8]:
# ============================================================================
# BƯỚC 5: CÁC TRÌNH TỐI ƯU HÓA BASELINE
# ============================================================================

def run_random_search(objective, search_space, n_trials):
    """Chạy Random Search (sử dụng Optuna)"""
    print("  Running Random Search...")
    sampler = optuna.samplers.RandomSampler()
    study = optuna.create_study(direction='maximize', sampler=sampler)
    
    # Hàm mục tiêu cho Optuna
    def optuna_objective(trial):
        params = {}
        for name, details in search_space.items():
            type = details[0]  # Type là phần tử đầu tiên của tuple
            if type == 'float':
                low, high = details[1], details[2]
                dist_type = details[3] if len(details) > 3 else None
                params[name] = trial.suggest_float(name, low, high, log=(dist_type == 'log'))
            elif type == 'int':
                low, high = details[1], details[2]
                params[name] = trial.suggest_int(name, low, high)
            elif type == 'categorical':
                choices = details[1]
                params[name] = trial.suggest_categorical(name, choices)
        return objective(params)

    study.optimize(optuna_objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_trial.value, study.best_params

def run_optuna_tpe(objective, search_space, n_trials):
    """Chạy Optuna TPE (baseline)"""
    print("  Running Optuna (TPE)...")
    sampler = optuna.samplers.TPESampler() # Sử dụng TPE sampler [17, 18, 19, 20]
    study = optuna.create_study(direction='maximize', sampler=sampler)
    
    # Hàm mục tiêu (giống hệt Random Search)
    def optuna_objective(trial):
        params = {}
        for name, details in search_space.items():
            type = details[0]  # Type là phần tử đầu tiên của tuple
            if type == 'float':
                low, high = details[1], details[2]
                dist_type = details[3] if len(details) > 3 else None
                params[name] = trial.suggest_float(name, low, high, log=(dist_type == 'log'))
            elif type == 'int':
                low, high = details[1], details[2]
                params[name] = trial.suggest_int(name, low, high)
            elif type == 'categorical':
                choices = details[1]
                params[name] = trial.suggest_categorical(name, choices)
        return objective(params)

    study.optimize(optuna_objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_trial.value, study.best_params

def run_hyperopt_tpe(objective, search_space, n_trials):
    """Chạy Hyperopt TPE (baseline)"""
    print("  Running Hyperopt (TPE)...")
    
    # 1. Chuyển đổi search_space sang định dạng Hyperopt
    hp_space = {}
    for name, details in search_space.items():
        type = details[0]  # Lấy phần tử đầu tiên của tuple
        
        if type == 'float':
            low, high = details[1], details[2]
            dist_type = details[3] if len(details) > 3 else None
            if dist_type == 'log':
                hp_space[name] = hp.loguniform(name, np.log(low), np.log(high))
            else:
                hp_space[name] = hp.uniform(name, low, high)
        elif type == 'int':
            low, high = details[1], details[2]
            # hp.quniform trả về float, nhưng được làm tròn theo 'q' (là 1). 
            # Chúng ta sẽ ép kiểu về int trong 'hyperopt_objective'
            hp_space[name] = hp.quniform(name, low, high, 1) 
        elif type == 'categorical':
            choices = details[1]
            hp_space[name] = hp.choice(name, choices)

    # 2. Định nghĩa hàm mục tiêu cho Hyperopt
    def hyperopt_objective(params):
        # Hyperopt trả về một số giá trị là float, cần ép kiểu về int
        # Cần kiểm tra lại logic này cho nhất quán
        params_copy = params.copy() # Làm việc trên bản sao để tránh lỗi
        for name, details in search_space.items():
            if details[0] == 'int' and name in params_copy:  # Sửa: kiểm tra type là phần tử đầu tiên
                params_copy[name] = int(params_copy[name])
                
        score = objective(params_copy)
        return {'loss': -score, 'status': STATUS_OK} # Hyperopt tối thiểu hóa

    # 3. Chạy fmin
    trials = Trials()
    best_raw = fmin(
        fn=hyperopt_objective,
        space=hp_space,
        algo=tpe.suggest, # [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
        max_evals=n_trials,
        trials=trials,
        verbose=0
    )
    
    best_score = -trials.best_trial['result']['loss']
    best_params = {}
    for name, val in best_raw.items():
        details = search_space[name]
        type = details[0]  # Type là phần tử đầu tiên của tuple
        
        if type == 'int':
            best_params[name] = int(val)
        elif type == 'categorical':
            choices = details[1]
            # `val` từ hyperopt cho `hp.choice` là một *chỉ số* (index)
            best_params[name] = choices[int(val)] 
        else: # float
            best_params[name] = val
    return best_score, best_params


def run_amsco_optimizer(objective, search_space, n_trials, slice_budget):
    """Chạy AMSCO (phương pháp của chúng ta)"""
    print("  Running AMSCO...")
    orchestrator = AMSCO_Orchestrator(
        objective_func=objective,
        search_space=search_space,
        total_budget=n_trials,
        slice_budget=slice_budget
    )
    result = orchestrator.run()
    return result['score'], result['params']

In [9]:
# ============================================================================
# BƯỚC 6: BỘ CÔNG CỤ THỰC NGHIỆM (EXPERIMENTAL HARNESS)
# ============================================================================

if __name__ == "__main__":
    
    # ----- CẤU HÌNH THỬ NGHIỆM -----
    DATASETS = ['breast_cancer', 'adult', 'telco_churn', 'credit_card_fraud']
    MODELS = ['logistic_regression', 'random_forest', 'lightgbm', 'xgboost']
    TOTAL_TRIALS = 50  # Giảm xuống để chạy demo nhanh, tăng lên (ví dụ: 100-200) cho kết quả thực tế
    SLICE_BUDGET = 10    # Budget cho mỗi lát cắt của AMSCO
    # --------------------------------
    
    results = []

    # Bắt đầu vòng lặp thử nghiệm
    for dataset_name in DATASETS:
        print(f"\n=======================================================")
        print(f"ĐANG THỬ NGHIỆM TRÊN BỘ DỮ LIỆU: {dataset_name.upper()}")
        print(f"=======================================================")
        
        # 1. Tải và chuẩn bị dữ liệu
        X, y, preprocessor, metric = get_data(dataset_name)
        if X is None:
            continue
            
        # Chia dữ liệu (chỉ dùng một phần nhỏ của Credit Card để chạy nhanh)
        if dataset_name == 'credit_card_fraud':
            # Lấy mẫu 10% để chạy demo nhanh hơn do bộ dữ liệu này rất lớn
            _, X, _, y = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
            print(f"  ... Lấy mẫu {len(y)} điểm từ Credit Card Fraud (do quá lớn).")

        for model_name in MODELS:
            print(f"\n{'-'*60}")
            print(f"Đang tối ưu mô hình: {model_name.upper()}")
            print(f"{'-'*60}")
            
            # 2. Lấy thông tin cụ thể cho thử nghiệm này
            search_space = MASTER_SEARCH_SPACES[model_name]
            objective_func = create_objective(X, y, model_name, preprocessor, metric)
            
            # Khởi tạo dictionary để lưu kết quả tạm thời
            model_results = {
                'scores': {},
                'times': {},
                'params': {}
            }
            
            # 3. Chạy các trình tối ưu hóa
            optimizers = [
                ('Random Search', run_random_search),
                ('Optuna (TPE)', run_optuna_tpe),
                ('Hyperopt (TPE)', run_hyperopt_tpe),
                ('AMSCO', run_amsco_optimizer)
            ]
            
            for optimizer_name, optimizer_func in optimizers:
                print(f"\nThực thi: {optimizer_name}...")
                start_time = time.time()
                
                if optimizer_name == 'AMSCO':
                    score, params = optimizer_func(objective_func, search_space, TOTAL_TRIALS, SLICE_BUDGET)
                else:
                    score, params = optimizer_func(objective_func, search_space, TOTAL_TRIALS)
                    
                exec_time = time.time() - start_time
                
                # Lưu kết quả
                model_results['scores'][optimizer_name] = score
                model_results['times'][optimizer_name] = exec_time
                model_results['params'][optimizer_name] = params
                
                # Thêm vào kết quả tổng hợp
                results.append({
                    'dataset': dataset_name,
                    'model': model_name,
                    'optimizer': optimizer_name,
                    'score': score,
                    'time': exec_time
                })
            
            # In bảng kết quả cho mô hình hiện tại sau khi hoàn thành tất cả các optimizer
            if all(optimizer_name in model_results['scores'] for optimizer_name, _ in optimizers):
                print(f"\n{'='*60}")
                print(f"KẾT QUẢ CHO {model_name.upper()}")
                print(f"{'='*60}")
                
                results_table = pd.DataFrame({
                    'Optimizer': list(model_results['scores'].keys()),
                    'Score': list(model_results['scores'].values()),
                    'Time (s)': list(model_results['times'].values())
                })
                
                # Định dạng các cột số
                results_table['Score'] = results_table['Score'].map('{:.4f}'.format)
                results_table['Time (s)'] = results_table['Time (s)'].map('{:.2f}'.format)
                
                # In bảng đã được định dạng
                print("\n" + results_table.to_string(index=False))
                print("\n" + "-"*60 + "\n")

    # In bảng kết quả tổng hợp
    print("\n\n" + "="*60)
    print(" "*20 + "KẾT QUẢ THỬ NGHIỆM TỔNG QUÁT" + " "*20)
    print("="*60 + "\n")
    
    # Chuyển kết quả thành DataFrame và thêm xử lý
    results_df = pd.DataFrame(results)
    
    # Tạo bảng tổng hợp theo dataset và model
    pivot_scores = results_df.pivot_table(
        values='score',
        index=['dataset', 'model'],
        columns='optimizer',
        aggfunc='mean'
    )
    
    # Tạo bảng tổng hợp thời gian
    pivot_times = results_df.pivot_table(
        values='time',
        index=['dataset', 'model'],
        columns='optimizer',
        aggfunc='mean'
    )
    
    # In kết quả điểm số
    print("ĐIỂM SỐ TRUNG BÌNH THEO DATASET VÀ MÔ HÌNH:")
    print("-"*60)
    print(pivot_scores.round(4).to_string())
    print("\n")
    
    # In kết quả thời gian
    print("THỜI GIAN CHẠY TRUNG BÌNH (GIÂY):")
    print("-"*60)
    print(pivot_times.round(2).to_string())
    print("\n")
    
    # Tính toán và in thống kê tổng hợp
    print("THỐNG KÊ TỔNG HỢP:")
    print("-"*60)
    stats_df = pd.DataFrame({
        'Optimizer': results_df['optimizer'].unique(),
        'Avg Score': [results_df[results_df['optimizer'] == opt]['score'].mean() for opt in results_df['optimizer'].unique()],
        'Std Score': [results_df[results_df['optimizer'] == opt]['score'].std() for opt in results_df['optimizer'].unique()],
        'Avg Time': [results_df[results_df['optimizer'] == opt]['time'].mean() for opt in results_df['optimizer'].unique()],
        'Total Time': [results_df[results_df['optimizer'] == opt]['time'].sum() for opt in results_df['optimizer'].unique()]
    })
    stats_df = stats_df.round(4)
    print(stats_df.to_string(index=False))
    
    print("\nGhi chú:")
    print("- Điểm số cao hơn là tốt hơn")
    print("- Thời gian được tính bằng giây")
    print("- Std Score: Độ lệch chuẩn của điểm số (càng thấp càng ổn định)")
    # 7. In kết quả cuối cùng
    print("\n\n=======================================================")
    print("         KẾT QUẢ THỬ NGHIỆM TỔNG QUÁT")
    print("=======================================================")
    
    results_df = pd.DataFrame(results)
    print(results_df.to_string())
    
    # Phân tích thống kê (bước tiếp theo)
    print("\nPhân tích thống kê (ví dụ: Friedman & Nemenyi): [30, 31, 32]")
    # (Tại đây, bạn có thể áp dụng các kiểm định thống kê trên results_df
    # để xem sự khác biệt có ý nghĩa thống kê hay không) [30, 31, 32]


ĐANG THỬ NGHIỆM TRÊN BỘ DỮ LIỆU: BREAST_CANCER
... Đang tải Breast Cancer

------------------------------------------------------------
Đang tối ưu mô hình: LOGISTIC_REGRESSION
------------------------------------------------------------

Thực thi: Random Search...
  Running Random Search...

Thực thi: Optuna (TPE)...
  Running Optuna (TPE)...

Thực thi: Hyperopt (TPE)...
  Running Hyperopt (TPE)...

Thực thi: AMSCO...
  Running AMSCO...
  [MetaController] Initializing Random
  [MetaController] Initializing Bayesian
  [MetaController] Initializing Grid
  [MetaController] UCB scores: {'Random': '2.11', 'Bayesian': '1.96', 'Grid': '1.98'} -> Chose Random
  [MetaController] UCB scores: {'Random': '1.72', 'Bayesian': '2.14', 'Grid': '2.17'} -> Chose Grid

KẾT QUẢ CHO LOGISTIC_REGRESSION

     Optimizer  Score Time (s)
 Random Search 0.9771     2.86
  Optuna (TPE) 0.9789     1.75
Hyperopt (TPE) 0.9771     2.10
         AMSCO 0.9771     1.73

------------------------------------------------

[W 2025-11-07 00:00:52,663] Trial 19 failed with parameters: {'classifier__n_estimators': 681, 'classifier__max_depth': 36, 'classifier__min_samples_split': 8, 'classifier__min_samples_leaf': 1} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\nguye\Downloads\AMSCO\venv\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\nguye\AppData\Local\Temp\ipykernel_2008\460436650.py", line 52, in optuna_objective
    return objective(params)
           ^^^^^^^^^^^^^^^^^
  File "C:\Users\nguye\AppData\Local\Temp\ipykernel_2008\838067945.py", line 39, in objective
    score = cross_val_score(pipeline, X, y, cv=cv, scoring=metric).mean()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\nguye\Downloads\AMSCO\venv\Lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args

KeyboardInterrupt: 